# Models overview

All models predict the mFRR marginal price of each auction (daily and weekly, upwards and downwards),
are retrained every month (walk-forward) and only use information published before the gate closure.

## Baseline

**Description:** the price of the last known auction of the same series.

* **Inputs:** last known log10 price of the same series (`lag1`)
* **Outputs:** `marginal_price_pred`

## Ridge

**Description:** linear regression with an L2 penalty (`src/models/ridge.py`). Before training, every input is rescaled to mean 0 and standard deviation 1 (on the training data), so that the penalty treats all inputs equally. Every month, one ridge is trained for each group (daily upwards, daily downwards, weekly upwards, weekly downwards).The variant **ridge_lags3_delta_per_series** trains one per series instead (14 models).

**Hyperparameters:** `alpha = 1.0`, group by `freq × direction`, target `level` (y) or
`delta` (y − lag1). Training auctions with missing lags are dropped, while auctions to predict are kept, as every auction needs a prediction: their missing lags are copied from the previous lag and, if none is known, filled with the training median.

* **ridge_allfeats_delta**
  * Inputs: `lag1`, `lag2`, `lag3`, `block_*`, `daytype_*`; target delta
  * Outputs: `marginal_price_pred`, `marginal_price_p10`, `marginal_price_p90`
* **ridge_lags3**
  * Inputs: `lag1`, `lag2`, `lag3`; target level
  * Outputs: `marginal_price_pred`, `marginal_price_p10`, `marginal_price_p90`
* **ridge_lags3_delta**
  * Inputs: `lag1`, `lag2`, `lag3`; target delta
  * Outputs: `marginal_price_pred`, `marginal_price_p10`, `marginal_price_p90`
* **ridge_lags3_delta_per_series**
  * Inputs: `lag1`, `lag2`, `lag3`; target delta; one model per series instead of per group
  * Outputs: `marginal_price_pred`, `marginal_price_p10`, `marginal_price_p90`

## Ridge quantiles

The ridge only predicts one value. The p10 and p90 are that prediction plus the 10% and 90% quantiles of
the model's errors on its training data. The offset is the same for every auction of a given model and month.

## LightGBM Quantiles

**Description:** gradient-boosted trees with the quantile (pinball) loss (`src/models/lgbm.py`). Three
models per group and month, for q = 0.1, 0.5, 0.9; the p50 is the prediction. Target delta (y − lag1), as they can only predict values seen in training, so a model of the price could not follow prices to new levels. The change from the last known price stays in a similar range every year, so predicting it and adding lag1 lets the forecast follow the level. The three predictions are sorted so that p10 ≤ p50 ≤ p90.

**Hyperparameters:** `learning_rate = 0.05`, `num_leaves = 15`, `min_child_samples = 50`,
`subsample = 0.8` (`subsample_freq = 1`), `colsample_bytree = 0.8`; number of trees chosen every month
among 100 / 300 / 600 on the last 3 months of the training data.

Outputs of every variant: `marginal_price_pred` (p50), `marginal_price_p10`, `marginal_price_p90`.

* **lgbm_quantile**
  * Inputs: `lag1`, `lag2`, `lag3`, `block_*`, `daytype_*`
* **lgbm_lagfeats**
  * Inputs: lgbm_quantile + `mom1` (lag1 − lag2), `mom2` (lag2 − lag3), `lag_mean`, `lag_std`
* **lgbm_ratio**
  * Inputs: lgbm_lagfeats + `ratio_lag1` (log10 offered / awarded of the last known auction)
* **lgbm_otherdir**
  * Inputs: lgbm_lagfeats + `other_lag1` (last known price of the opposite direction), `other_gap` (other_lag1 − lag1)
* **lgbm_weekly**
  * Inputs: lgbm_lagfeats + `weekly_lag1` (last known weekly price, same direction), `weekly_gap` (weekly_lag1 − lag1)
* **lgbm_dayahead**
  * Inputs: lgbm_otherdir + `da_ch_last`, `da_ch_mean7`, `da_ch_block` (Swiss day-ahead prices, known from 13:00 the day before)
* **lgbm_reservoirs**
  * Inputs: lgbm_otherdir + `res_fill`, `res_change4`, `res_anomaly` (Swiss reservoir levels,weekly value of Sunday 24h, treated as known 7 days later)

## Training

- **Monthly walk-forward** (`WalkForwardSplitter`): before each test month, every model is retrained on all
  auctions already published at the start of that month (`avail` before the month starts). The test month
  is predicted, then the window moves one month forward.
- **One model per group** (`freq × direction`) and month; the ridge variant `per_series` uses one per series.
- **Periods:** models and settings are compared and chosen on the validation period (2022–2024);
  2025 is predicted only once, in the evaluation.

## Metrics

Computed per group, on auctions with a price (no-buy auctions are predicted but not scored), on log10 prices:

- **MAE** of the prediction (p50).
- **RMSE:** like the MAE, but large errors weigh more.
- **Pinball loss** of q10, q50 and q90: weighted on the side where it should rarely be wrong. For q10, a prediction above the actual price costs 9 times more than one below, so the best p10 is the price exceeded about 90% of the time; q50 weighs both sides equally (half the MAE); q90 is the mirror of q10.
- **Coverage 80%:** share of actual prices between p10 and p90 (target 0.80).

Bidding simulation (1 MW in every auction, pay-as-bid, CHF):

- **Revenue** per year: a bid is paid if it is at or below the marginal price; no-buy auctions pay nothing.
- **Share of the perfect revenue:** revenue divided by the revenue of bidding exactly the
  marginal price.
- **Share of accepted bids.**